# TP1bis - Incidents Data Processing Pipeline
## Bronze to Silver Transformation with Quality Assurance

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path

# Configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 2. Load and Explore Bronze Data

In [ ]:
# Load data
bronze = pd.read_csv('releves_incidents_anonymised.csv')

# Basic info
print(f"Dimensions: {bronze.shape}")
print(f"\nColumns: {list(bronze.columns)}")
print(f"\nFirst 10 rows:")
print(bronze.head(10))

In [ ]:
# Check for missing values
print("NaN par colonne:")
print(bronze.isna().sum())
print(f"\nTotal NaN: {bronze.isna().sum().sum()}")

## 3. Analyze Data Distribution and Outliers

In [ ]:
# Descriptive statistics
bronze.describe()

In [ ]:
# Get numeric columns dynamically
numeric_cols = bronze.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

## 4. Create Ingestion Directory Structure

In [ ]:
def create_ingestion_dir(base: str = "artifacts/ingestions", topic: str = "incidents") -> Path:
    """
    Create versioned ingestion directory with timestamp.
    
    Parameters
    ----------
    base : str
        Root artifacts path
    topic : str
        Subject folder name
    
    Returns
    -------
    Path
        Full directory path
    """
    timestamp = datetime.now().strftime("%Y%m%d%H%M")
    run_dir = Path(base) / topic / timestamp
    run_dir.mkdir(parents=True, exist_ok=True)
    print(f"Ingestion directory created: {run_dir}")
    return run_dir

run_dir = create_ingestion_dir()

## 5. Generate Exploratory Visualizations

In [ ]:
# Distribution of numeric variables
if numeric_cols:
    n_cols = len(numeric_cols)
    n_rows = (n_cols + 2) // 3
    
    bronze[numeric_cols].hist(bins=50, figsize=(15, n_rows*4), layout=(n_rows, 3))
    plt.suptitle('Distribution of Bronze Variables')
    plt.tight_layout()
    plt.savefig(run_dir / 'distribution_bronze.png', bbox_inches='tight', dpi=100)
    plt.show()

In [ ]:
# Boxplots to detect outliers visually
if numeric_cols and 'machine_id' in bronze.columns:
    n_cols = len(numeric_cols)
    n_rows = (n_cols + 2) // 3
    
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, n_rows*4))
    axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols):
        sns.boxplot(data=bronze, x='machine_id', y=col, ax=axes[i])
        axes[i].set_title(col)
        axes[i].tick_params(axis='x', rotation=45)
    
    # Hide unused subplots
    for j in range(n_cols, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Boxplots by Machine - Bronze')
    plt.tight_layout()
    plt.savefig(run_dir / 'boxplots_bronze.png', bbox_inches='tight', dpi=100)
    plt.show()

## 6. Detect and Handle Outliers Using IQR

In [ ]:
# Detect outliers using IQR method
print("="*70)
print("OUTLIER DETECTION (IQR Method - 3*IQR bounds)")
print("="*70)

outlier_summary = {}

for col in numeric_cols:
    Q1 = bronze[col].quantile(0.25)
    Q3 = bronze[col].quantile(0.75)
    IQR = Q3 - Q1
    low = Q1 - 3 * IQR
    high = Q3 + 3 * IQR
    
    mask = (bronze[col] < low) | (bronze[col] > high)
    n_outliers = mask.sum()
    outlier_summary[col] = n_outliers
    
    print(f"{col:30s} → {n_outliers:4d} outliers (bounds: [{low:10.2f}, {high:10.2f}])")

print("="*70)
print(f"Total outliers in Bronze: {sum(outlier_summary.values())}")

## 7. Transform Bronze to Silver Layer

In [ ]:
# Create silver layer
silver = bronze.copy()

# 1. Type conversion
if 'machine_id' in silver.columns:
    silver['machine_id'] = silver['machine_id'].astype('category')

# 2. Parse datetime if date/time columns exist
if 'date' in silver.columns and 'time' in silver.columns:
    silver['datetime'] = pd.to_datetime(silver['date'] + ' ' + silver['time'])
elif 'datetime' in silver.columns:
    silver['datetime'] = pd.to_datetime(silver['datetime'])
elif 'timestamp' in silver.columns:
    silver['timestamp'] = pd.to_datetime(silver['timestamp'])

# 3. Sort chronologically
datetime_col = None
if 'datetime' in silver.columns:
    datetime_col = 'datetime'
elif 'timestamp' in silver.columns:
    datetime_col = 'timestamp'

if datetime_col and 'machine_id' in silver.columns:
    silver = silver.sort_values(['machine_id', datetime_col]).reset_index(drop=True)

# 4. Temporal features
if datetime_col:
    silver['year'] = silver[datetime_col].dt.year
    silver['month'] = silver[datetime_col].dt.month
    silver['day'] = silver[datetime_col].dt.day
    silver['hour'] = silver[datetime_col].dt.hour
    silver['day_of_week'] = silver[datetime_col].dt.dayofweek
    silver['is_weekend'] = silver['day_of_week'].isin([5, 6])

# 5. Handle outliers - replace with NaN, then interpolate per machine
IQR_FACTOR = 3
outlier_report = {}

for col in numeric_cols:
    Q1 = silver[col].quantile(0.25)
    Q3 = silver[col].quantile(0.75)
    IQR = Q3 - Q1
    low = Q1 - IQR_FACTOR * IQR
    high = Q3 + IQR_FACTOR * IQR
    
    mask = (silver[col] < low) | (silver[col] > high)
    outlier_report[col] = int(mask.sum())
    silver.loc[mask, col] = np.nan

# 6. Linear interpolation per machine
if 'machine_id' in silver.columns:
    silver[numeric_cols] = (
        silver.groupby('machine_id', observed=True)[numeric_cols]
        .transform(lambda g: g.interpolate(method='linear', limit_direction='both'))
    )

# 7. Round numeric values
silver[numeric_cols] = silver[numeric_cols].round(2)

print(f"Silver shape: {silver.shape}")
print(f"NaN remaining: {silver.isna().sum().sum()}")
print(f"\nOutliers treated:")
for col, n in outlier_report.items():
    print(f"  {col}: {n}")

In [ ]:
# Export silver CSV
silver_output_path = run_dir / 'releves_incidents_silver.csv'
silver.to_csv(silver_output_path, index=False)

print(f"Silver layer exported to: {silver_output_path}")

## 8. Compare Before and After Outlier Treatment

In [ ]:
# Compare outlier counts
print("\n" + "="*70)
print("OUTLIER COMPARISON: BRONZE vs SILVER")
print("="*70)

print("\nBRONZE (before treatment):")
print("-"*70)
bronze_outliers = {}
for col in numeric_cols:
    Q1 = bronze[col].quantile(0.25)
    Q3 = bronze[col].quantile(0.75)
    IQR = Q3 - Q1
    low = Q1 - 3 * IQR
    high = Q3 + 3 * IQR
    n = ((bronze[col] < low) | (bronze[col] > high)).sum()
    bronze_outliers[col] = n
    print(f"{col:30s} → {n:6d} outliers")

print("\nSILVER (after interpolation):")
print("-"*70)
silver_outliers = {}
for col in numeric_cols:
    Q1 = silver[col].quantile(0.25)
    Q3 = silver[col].quantile(0.75)
    IQR = Q3 - Q1
    low = Q1 - 3 * IQR
    high = Q3 + 3 * IQR
    n = ((silver[col] < low) | (silver[col] > high)).sum()
    silver_outliers[col] = n
    print(f"{col:30s} → {n:6d} outliers")

print("\n" + "="*70)
total_bronze = sum(bronze_outliers.values())
total_silver = sum(silver_outliers.values())
reduction = total_bronze - total_silver

print(f"Total outliers BRONZE:    {total_bronze}")
print(f"Total outliers SILVER:    {total_silver}")
print(f"Outliers eliminated:      {reduction}")
print(f"Reduction percentage:     {100*reduction/total_bronze:.1f}%" if total_bronze > 0 else "N/A")
print("="*70)